In [1]:
!pip install anthropic --quiet

import os
import json
import anthropic
import numpy as np
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv
from supabase import create_client, Client
from sklearn.ensemble import IsolationForest, GradientBoostingRegressor
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

load_dotenv(dotenv_path="/Users/mohamedmunir/Desktop/FinScope-Ai/ai-service/notebooks/.env", override=True)

supabase: Client = create_client(os.getenv("SUPABASE_URL"), os.getenv("SUPABASE_SERVICE_KEY"))
claude           = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

print("Connected to Supabase and Claude")


Connected to Supabase and Claude


In [2]:
# Tools the agent can call
TOOLS = [
    {
        "name": "pull_data",
        "description": "Pull all transactions and user finances from Supabase",
        "input_schema": {"type": "object", "properties": {}, "required": []}
    },
    {
        "name": "clean_data",
        "description": "Clean and prepare transaction data for ML models",
        "input_schema": {
            "type": "object",
            "properties": {
                "filter_type": {
                    "type": "string",
                    "description": "Filter by transaction type: expense, deposit, all",
                    "enum": ["expense", "deposit", "all"]
                }
            },
            "required": ["filter_type"]
        }
    },
    {
        "name": "run_anomaly_detection",
        "description": "Run Isolation Forest to detect anomalous transactions",
        "input_schema": {
            "type": "object",
            "properties": {
                "contamination": {
                    "type": "number",
                    "description": "Expected proportion of anomalies (0.05 to 0.3)"
                }
            },
            "required": ["contamination"]
        }
    },
    {
        "name": "run_clustering",
        "description": "Run KMeans to segment users into spending personas",
        "input_schema": {
            "type": "object",
            "properties": {
                "n_clusters": {
                    "type": "integer",
                    "description": "Number of user clusters (2 to 5)"
                }
            },
            "required": ["n_clusters"]
        }
    },
    {
        "name": "run_forecast",
        "description": "Run Gradient Boosting to forecast next 7 days of spending",
        "input_schema": {"type": "object", "properties": {}, "required": []}
    },
    {
        "name": "write_insights_to_supabase",
        "description": "Write all ML insights and recommendations back to Supabase",
        "input_schema": {
            "type": "object",
            "properties": {
                "insights": {
                    "type": "object",
                    "description": "Dict of insights to store"
                }
            },
            "required": ["insights"]
        }
    },
    {
        "name": "generate_report",
        "description": "Generate a final natural language report summarising all findings",
        "input_schema": {
            "type": "object",
            "properties": {
                "findings": {
                    "type": "object",
                    "description": "All ML findings to summarise"
                }
            },
            "required": ["findings"]
        }
    }
]

print(f"Defined {len(TOOLS)} agent tools")

# git commit -m "# Tools the agent can call
TOOLS = [
    {
        "name": "pull_data",
        "description": "Pull all transactions and user finances from Supabase",
        "input_schema": {"type": "object", "properties": {}, "required": []}
    },
    {
        "name": "clean_data",
        "description": "Clean and prepare transaction data for ML models",
        "input_schema": {
            "type": "object",
            "properties": {
                "filter_type": {
                    "type": "string",
                    "description": "Filter by transaction type: expense, deposit, all",
                    "enum": ["expense", "deposit", "all"]
                }
            },
            "required": ["filter_type"]
        }
    },
    {
        "name": "run_anomaly_detection",
        "description": "Run Isolation Forest to detect anomalous transactions",
        "input_schema": {
            "type": "object",
            "properties": {
                "contamination": {
                    "type": "number",
                    "description": "Expected proportion of anomalies (0.05 to 0.3)"
                }
            },
            "required": ["contamination"]
        }
    },
    {
        "name": "run_clustering",
        "description": "Run KMeans to segment users into spending personas",
        "input_schema": {
            "type": "object",
            "properties": {
                "n_clusters": {
                    "type": "integer",
                    "description": "Number of user clusters (2 to 5)"
                }
            },
            "required": ["n_clusters"]
        }
    },
    {
        "name": "run_forecast",
        "description": "Run Gradient Boosting to forecast next 7 days of spending",
        "input_schema": {"type": "object", "properties": {}, "required": []}
    },
    {
        "name": "write_insights_to_supabase",
        "description": "Write all ML insights and recommendations back to Supabase",
        "input_schema": {
            "type": "object",
            "properties": {
                "insights": {
                    "type": "object",
                    "description": "Dict of insights to store"
                }
            },
            "required": ["insights"]
        }
    },
    {
        "name": "generate_report",
        "description": "Generate a final natural language report summarising all findings",
        "input_schema": {
            "type": "object",
            "properties": {
                "findings": {
                    "type": "object",
                    "description": "All ML findings to summarise"
                }
            },
            "required": ["findings"]
        }
    }
]

print(f"Defined {len(TOOLS)} agent tools")



Defined 7 agent tools
